# Production Data Validation and Quality

This project focuses on checking the accuracy and consistency of production-related data. The dataset contains records from manufacturing processes, including production output, defects, costs, and operating conditions.

The goal of the project is to identify missing values and logical inconsistencies that could affect the reliability of the data. SQL queries are used to explore the data, highlight potential issues, and summarize findings so the data can be corrected before being used for reporting or analysis.

## Retrieve data

In [51]:
import pandas as pd
import sqlite3

# convert from .csv to .sqlite format
csv_file = "manufacturing_data.csv"
df = pd.read_csv(csv_file)
df.columns = df.columns.str.replace(" ", "_")
con = sqlite3.connect("production.sqlite")
df.to_sql("production_temp", con, if_exists="replace", index=False)

cur = con.cursor()

In [52]:
columns = cur.execute("PRAGMA table_info(production_temp);").fetchall()

for column in columns:
    column_name = column[1]
    column_type = column[2]
    print(f"{column_name:<30}{column_type:>10}")

Production_ID                    INTEGER
Date                                TEXT
Product_Type                        TEXT
Machine_ID                       INTEGER
Shift                               TEXT
Units_Produced                   INTEGER
Defects                             REAL
Production_Time_Hours               REAL
Material_Cost_Per_Unit              REAL
Labour_Cost_Per_Hour                REAL
Energy_Consumption_kWh              REAL
Operator_Count                   INTEGER
Maintenance_Hours                   REAL
Down_time_Hours                     REAL
Production_Volume_Cubic_Meters      REAL
Scrap_Rate                          REAL
Rework_Hours                        REAL
Quality_Checks_Failed            INTEGER
Average_Temperature_C               REAL
Average_Humidity_Percent            REAL


There is a total of 20 columns in our dataset. Most of them contain real numbers, while some store integer or text values.

*Production_ID* is likely the row identifier for the data. However, *Pandas* to_sql() method created a table without specifying a primary key, so it would be a good idea to make sure there are no duplicates or null values in the *Production_ID* column and then set it as the primary key. Unfortunately, SQLite does not support altering the table to set a primary key after it has been created, but we are still going to perform a check.

## Primary Key Validation

In [53]:
# check if Production_ID uniquely identifies each row as intended

res = cur.execute("""
    SELECT Production_ID, COUNT(*)
    FROM production 
    GROUP BY Production_ID
    HAVING COUNT(*) > 1;
    """).fetchall()

res

[]

In [54]:
# check for NULL values in Production_ID

res = cur.execute("""
    SELECT *
    FROM production
    WHERE production_id is NULL
    """).fetchall()

res

[]

There are no problems, so we can proceed.

## Missing Values

In [55]:
cur.execute(f"PRAGMA table_info({"production"})")
columns = [info[1] for info in cur.fetchall()]  # info[1] is column name

# Dictionary to store missing value counts
missing_counts = {}

# Loop through columns and count NULLs
for col in columns:
    query = f"SELECT COUNT(*) FROM production WHERE {col} IS NULL"
    cur.execute(query)
    count = cur.fetchone()[0]
    if count > 0:
        missing_counts[col] = count

missing_df = pd.DataFrame(list(missing_counts.items()), columns=["Column", "Missing_Count"])
print(missing_df)

              Column  Missing_Count
0            Defects            299
1  Maintenance_Hours            300
2    Down_time_Hours            300
3       Rework_Hours            300


It looks like the only missing values in the dataset come from these four columns: *defects*, *maintenance hours*, *down time hours* and *rework hours*. The amount is basically the same for each column, so we may be tempted to say that these missing values occur in the same rows across the columns. After all, if a machine did not need maintenance, it also did not experience a downtime: it would make sense.

Let's check if our intuition is correct.

In [56]:
# I decided to ignore the Defects column since it has got 299 missing values instead of 300
# the first sum counts the number of rows where all of the three other columns have missing values
# the second sum counts the number of rows where any of the three other columns have missing values

res = cur.execute("""
    SELECT
    SUM(CASE WHEN Maintenance_Hours IS NULL AND Down_Time_Hours IS NULL AND Rework_Hours IS NULL THEN 1 ELSE 0 END) AS all_missing_same_rows,
    SUM(CASE WHEN Maintenance_Hours IS NULL OR Down_Time_Hours IS NULL OR Rework_Hours IS NULL THEN 1 ELSE 0 END) AS any_missing_rows
    FROM production;
    """).fetchall()

res

[(3, 812)]

It turns out that actually there are only three rows where all of the values are missing: the other missing values are scattered in different rows across the dataset. This makes handling the data a bit more problematic.

We could encode the missing values as 0, but that would not necessarily be the best solution. We may exclude the rows altogether, but that would mean to lose a considerable amount of data. For now, the best thing is probably to report the problem and let a domain expert decide what to do.

## Invalid values

We are going to check for some invalid values, such as a negative number of units produced or a scrap rate not bounded between 0 and 1.

In [57]:
# check for a negative number of units produced

res = cur.execute("""
    SELECT *
    FROM production
    WHERE Units_Produced < 0
    """).fetchall()

res

[]

In [58]:
# check for a negative production time

res = cur.execute("""
    SELECT *
    FROM production
    WHERE Production_Time_Hours < 0
    """).fetchall()

res

[]

In [59]:
# check for cases where the number of defects is greater than the number of units produced

res = cur.execute("""
    SELECT *
    FROM production
    WHERE Defects > Units_Produced;
    """).fetchall()

res

[]

In [60]:
# check for a scrap rate lower than 0 or greater than 1

res = cur.execute("""
    SELECT *
    FROM production
    WHERE Scrap_Rate < 0 OR Scrap_Rate > 1;
    """).fetchall()

res

[]

Everything seems to be fine.

# Additional insights

In [61]:
# check which product types exerience more down time hours than production time hours

res = cur.execute("""
SELECT
Product_Type,
COUNT(*) AS rows_with_downtime_gt_production
FROM production
WHERE Down_Time_Hours > Production_Time_Hours
GROUP BY Product_Type
ORDER BY COUNT(*) DESC;
""").fetchall()

res

[('Textiles', 21),
 ('Electronics', 19),
 ('Appliances', 15),
 ('Automotive', 14),
 ('Furniture', 12)]

The product type with the most occurrences of a down time greater than the production time is textile: we may suppose that the machines used to produce textile products require more maintenance and are more likely to experience runs with a down time which exceeds the production time.

## Conclusions

We have performed some basic checks on the production data we had available: we have made sure that the primary key is unique, looked for missing, unusual or invalid values, and tried to interpret some patterns in the data.